# Tool-Router Ladder: LoRA SFT (V1-09)

**Settings:** Accelerator **GPU T4**, Internet **on**, and the private dataset attached (Add Input). The dataset holds `train.jsonl`, `manifest.json` and the `prompts-*.jsonl` files.

**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked for this notebook.

**Do not install vLLM in this notebook.** It replaces torch. Generation has its own notebook.

Order of cells:
1. Smoke: one step on the longest episode. This proves the 12,176-token worst case fits in T4 memory.
2. 0.5B.
3. 1.5B.

Each run checkpoints every epoch to the Hub and resumes if the session dies.

**Send back:** the two Hub repo ids and the last `[train] done: ...` line of each run.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])

In [ ]:
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline
!pip install -q peft

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "CHANGE-ME"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

### 1. Smoke: about 3 minutes including the model download. If this runs out of memory, stop and send the error.

In [ ]:
!python src/train.py --config configs/qwen05b.yaml --data "$DATA" --output-dir /kaggle/working/smoke --smoke

### 2. Qwen2.5-0.5B: the first real run tells us the time; about 21 optimizer steps.

In [ ]:
!python src/train.py --config configs/qwen05b.yaml --data "$DATA" --output-dir /kaggle/working/qwen05b --hub-repo "$HF_USER/tool-router-qwen05b-sft" --wandb

### 3. Qwen2.5-1.5B

In [ ]:
!python src/train.py --config configs/qwen15b.yaml --data "$DATA" --output-dir /kaggle/working/qwen15b --hub-repo "$HF_USER/tool-router-qwen15b-sft" --wandb